# 05 — Self-Healing Incident (v1.0)

Reproduz o fluxo completo de self-healing / root-cause-analysis descrito em
[`escopo.md`](../escopo.md) (Incident #017):

```
Serviço web caiu
   |
Health Check detecta falha
   |
Coleta logs + CPU + RAM + portas
   |
Diagnóstico
   |
Executa remediation playbook
   |
Reinicia serviço
   |
Valida recuperação
   |
Gera relatório do incidente
```

Referência do escopo:

```
Incident #017
Host: linux-web-02
Problem: nginx unavailable
Root cause: service stopped
Action: restart via Ansible
Validation: HTTP 200
Recovery time: 13s
Status: RESOLVED
```

Aqui, o "serviço" é modelado localmente como um processo `python -m
http.server` (mesma técnica documentada em `service_check.py`), para que o
incidente inteiro — queda real, detecção real, remediação real, validação
real — rode de ponta a ponta sem precisar de um host Linux remoto.

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


In [2]:
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path as _Path

from automation.troubleshooting.health_check import check_port, find_free_port
from automation.troubleshooting.service_check import check_service_status
from automation.troubleshooting.disk_check import check_disk_usage
from automation.troubleshooting.remediation import restart_local_process, decide_remediation

HOST = "linux-web-02"  # nome simbólico do host, como no escopo (Incident #017)
ENVIRONMENT = "dev"
SERVICE_NAME = "nginx"  # nome simbólico do serviço (processo real: python http.server)

# Webroot minúsculo e descartável -- serve um único index.html em vez da
# árvore inteira do projeto, para as checagens HTTP responderem instantaneamente.
WEBROOT = _Path(tempfile.mkdtemp(prefix="nb05_webroot_"))
(WEBROOT / "index.html").write_text("<html><body>demo ok</body></html>", encoding="utf-8")

33

## Passo 1 — Provisiona o serviço "nginx" (processo local descartável) e o derruba

In [3]:
port = find_free_port()

def start_service():
    return subprocess.Popen(
        [sys.executable, "-m", "http.server", str(port), "--bind", "127.0.0.1"],
        cwd=str(WEBROOT), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )

service_proc = start_service()
for _ in range(50):
    if check_port(host="127.0.0.1", port=port, timeout=0.2):
        break
    time.sleep(0.1)

print(f"Serviço '{SERVICE_NAME}' (pid={service_proc.pid}) no ar em 127.0.0.1:{port}")

# Simula a queda: mata o processo (equivalente a "nginx unavailable" no escopo).
service_proc.kill()
service_proc.wait(timeout=5)
print(f"Incidente provocado: processo {service_proc.pid} finalizado — serviço indisponível.")

Serviço 'nginx' (pid=19292) no ar em 127.0.0.1:60533
Incidente provocado: processo 19292 finalizado — serviço indisponível.


## Passo 2 — Health Check detecta a falha + coleta de diagnóstico (CPU/RAM/portas/disco)

In [4]:
incident_started_at = datetime.now(timezone.utc)

diagnosis = check_service_status(
    service_name=SERVICE_NAME,
    host="127.0.0.1",
    port=port,
    process_name="http.server",
)
print("Diagnóstico (health/service check):")
for k, v in diagnosis.items():
    print(f"  {k}: {v}")

disk = check_disk_usage(path=str(PROJECT_ROOT.anchor or "/"))
print(f"\nDisco (contexto adicional do diagnóstico): {disk['percent_used']}% usado (alert={disk['alert']})")

try:
    import psutil
    cpu_percent = psutil.cpu_percent(interval=0.3)
    ram_percent = psutil.virtual_memory().percent
    print(f"CPU: {cpu_percent}%  |  RAM: {ram_percent}%")
except ImportError:
    cpu_percent = ram_percent = None
    print("psutil indisponível — pulando coleta de CPU/RAM (não bloqueia o restante do fluxo).")

Diagnóstico (health/service check):
  service_name: nginx
  host: 127.0.0.1
  port: 60533
  checked_at: 2026-08-19T23:24:02.324314+00:00
  port_open: False
  process_name: http.server
  process_running: False
  status: down

Disco (contexto adicional do diagnóstico): 73.63% usado (alert=False)


CPU: 100.0%  |  RAM: 47.8%


## Passo 3 — Diagnóstico de causa raiz (root cause)

In [5]:
root_cause = "service stopped" if diagnosis["status"] == "down" else "unknown"
remediation_action = decide_remediation(diagnosis)
print(f"Root cause: {root_cause}")
print(f"Ação de remediação decidida (regra automation/troubleshooting/remediation.py::decide_remediation): {remediation_action}")

Root cause: service stopped
Ação de remediação decidida (regra automation/troubleshooting/remediation.py::decide_remediation): restart_local_process


## Passo 4 — Executa remediação

No escopo real, a remediação de um host Linux roda via Ansible
(`run_ansible_playbook`). Aqui tentamos o caminho real primeiro (útil se
`ansible-playbook` estiver disponível via WSL) e, como fallback gracioso e
sempre funcional, usamos `restart_local_process` — o mesmo mecanismo que o
módulo de remediação usa para o demo local, reiniciando o processo e
validando a porta.

In [6]:
ansible_playbook_path = PROJECT_ROOT / "ansible" / "playbooks" / "restart_service.yml"
remediation_via = "restart_local_process"

if remediation_action == "restart_local_process":
    if shutil.which("ansible-playbook") and ansible_playbook_path.exists():
        from automation.troubleshooting.remediation import run_ansible_playbook
        ansible_result = run_ansible_playbook(str(ansible_playbook_path))
        print(f"Tentativa via Ansible: success={ansible_result['success']}")
        if not ansible_result["success"]:
            print(f"  (fallback para restart local) {ansible_result.get('error', '')}")
    else:
        print("ansible-playbook não disponível neste ambiente (ou playbook ainda não existe) "
              "— usando restart_local_process, que é o mecanismo real usado pelo demo de "
              "self-healing local.")

    remediation_result = restart_local_process(
        start_command=[sys.executable, "-m", "http.server", str(port), "--bind", "127.0.0.1"],
        port=port,
        host="127.0.0.1",
        cwd=str(WEBROOT),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"\nRemediação: {remediation_result['action']}")
    print(f"  success: {remediation_result['success']}")
    print(f"  novo pid: {remediation_result['pid']}")
    print(f"  recovery_time_seconds: {remediation_result['recovery_time_seconds']}")
else:
    remediation_result = None
    print("Nenhuma remediação necessária segundo o diagnóstico.")

ansible-playbook não disponível neste ambiente (ou playbook ainda não existe) — usando restart_local_process, que é o mecanismo real usado pelo demo de self-healing local.



Remediação: restarted local process: C:\Users\Yuri_\AppData\Local\Programs\Python\Python310\python.exe -m http.server 60533 --bind 127.0.0.1
  success: True
  novo pid: 3908
  recovery_time_seconds: 3.859


## Passo 5 — Valida recuperação (novo health check)

In [7]:
validation = check_service_status(
    service_name=SERVICE_NAME,
    host="127.0.0.1",
    port=port,
    process_name="http.server",
)
print("Validação pós-remediação:")
for k, v in validation.items():
    print(f"  {k}: {v}")

import urllib.request
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=3) as resp:
        http_status = resp.status
except Exception as exc:
    http_status = None
    print(f"Não foi possível obter o status HTTP (segue com validation acima): {exc}")

validation_str = f"HTTP {http_status}" if http_status else validation["status"].upper()
print(f"\nValidation: {validation_str}")

Validação pós-remediação:
  service_name: nginx
  host: 127.0.0.1
  port: 60533
  checked_at: 2026-08-19T23:24:09.881471+00:00
  port_open: True
  process_name: http.server
  process_running: True
  status: up



Validation: HTTP 200


## Passo 6 — Gera o relatório do incidente (formato do escopo — Incident #017)

In [8]:
incident_resolved_at = datetime.now(timezone.utc)
recovery_time_seconds = (
    remediation_result["recovery_time_seconds"] if remediation_result else 0.0
)
status = "RESOLVED" if validation["status"] == "up" else "UNRESOLVED"

incident_report = {
    "incident_id": 17,
    "host": HOST,
    "environment": ENVIRONMENT,
    "problem": f"{SERVICE_NAME} unavailable",
    "root_cause": root_cause,
    "action": remediation_result["action"] if remediation_result else "no action taken",
    "validation": validation_str,
    "recovery_time_seconds": recovery_time_seconds,
    "status": status,
    "created_at": incident_started_at.isoformat(),
    "resolved_at": incident_resolved_at.isoformat(),
}

print("=" * 50)
print(f"Incident #{incident_report['incident_id']:03d}")
print("=" * 50)
print(f"Host: {incident_report['host']}")
print(f"Problem: {incident_report['problem']}")
print(f"Root cause: {incident_report['root_cause']}")
print(f"Action: {incident_report['action']}")
print(f"Validation: {incident_report['validation']}")
print(f"Recovery time: {incident_report['recovery_time_seconds']}s")
print(f"Status: {incident_report['status']}")

Incident #017
Host: linux-web-02
Problem: nginx unavailable
Root cause: service stopped
Action: restarted local process: C:\Users\Yuri_\AppData\Local\Programs\Python\Python310\python.exe -m http.server 60533 --bind 127.0.0.1
Validation: HTTP 200
Recovery time: 3.859s
Status: RESOLVED


In [9]:
# Mapeia o relatório para o contrato de dados da tabela `incidents`
# (database/schema.sql), pronto para um INSERT real quando o MySQL estiver
# disponível (ver Notebook 04, Parte 3).
insert_sql = """
INSERT INTO incidents
  (host, environment, problem, root_cause, action, validation, recovery_time_seconds, status, created_at)
VALUES
  (%s, %s, %s, %s, %s, %s, %s, %s, %s);
""".strip()
print("SQL que seria executado contra database/schema.sql::incidents:\n")
print(insert_sql)
print("\nParâmetros:")
print((
    incident_report["host"], incident_report["environment"], incident_report["problem"],
    incident_report["root_cause"], incident_report["action"], incident_report["validation"],
    incident_report["recovery_time_seconds"], incident_report["status"], incident_report["created_at"],
))

SQL que seria executado contra database/schema.sql::incidents:

INSERT INTO incidents
  (host, environment, problem, root_cause, action, validation, recovery_time_seconds, status, created_at)
VALUES
  (%s, %s, %s, %s, %s, %s, %s, %s, %s);

Parâmetros:
('linux-web-02', 'dev', 'nginx unavailable', 'service stopped', 'restarted local process: C:\\Users\\Yuri_\\AppData\\Local\\Programs\\Python\\Python310\\python.exe -m http.server 60533 --bind 127.0.0.1', 'HTTP 200', 3.859, 'RESOLVED', '2026-08-19T23:24:00.633239+00:00')


In [10]:
# Limpeza final: garante que nenhum processo/arquivo de demonstração fica pendurado.
new_pid = remediation_result["pid"] if remediation_result else None
if new_pid:
    try:
        import psutil
        psutil.Process(new_pid).kill()
        print(f"Processo de demonstração {new_pid} finalizado.")
    except Exception as exc:
        print(f"Não foi possível finalizar o processo {new_pid} (pode já ter encerrado): {exc}")

shutil.rmtree(WEBROOT, ignore_errors=True)
print(f"Webroot temporário removido: {WEBROOT}")

Processo de demonstração 3908 finalizado.
Webroot temporário removido: C:\Users\Yuri_\AppData\Local\Temp\nb05_webroot_3i__513t


## Resumo

O incidente completo — queda real de um processo local, detecção via
`service_check`, diagnóstico via `disk_check`/`psutil`, decisão de
remediação via `decide_remediation`, execução via `restart_local_process`
(com tentativa real de Ansible quando disponível), e validação via um novo
health check + requisição HTTP real — rodou de ponta a ponta e gerou um
relatório no mesmo formato do `Incident #017` do escopo, com `recovery_time_seconds`
medido de verdade (não simulado) e pronto para persistir na tabela
`incidents` do MySQL (Notebook 04).